# Hate Speech Bias Detection Demo
**English + Hindi Models** | COL8381 Project  
Compare zero-shot, fine-tuned, and debiased models across Indian (ULI) and Western (HateXplain) datasets.

In [1]:
ls "/kaggle/input/datasets/sansahaiiitd/all-models/all_models"

muril_hindi_finetuned_model/  roberta_uli_zeroshot.pth
muril_hindi_zeroshot_model/   roberta_west_finetuned.pth
roberta_uli_debiased.pth      roberta_west_zeroshot.pth
roberta_uli_finetuned.pth


In [2]:
# ── Install dependencies ───────────────────────────────────────────────
!pip install -q gradio lime transformers torch
print('Dependencies installed!')

Dependencies installed!


In [3]:
# ── SETUP: Device & Model Paths ─────────────────────────────────────────
import torch
import numpy as np
import gradio as gr
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, pipeline
)
from lime.lime_text import LimeTextExplainer
import warnings
warnings.filterwarnings('ignore')

DEVICE = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU" if DEVICE == 0 else "CPU"}')

# ─────────────────────────────────────────────────────────────────────────
# MODEL CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────

# English models (RoBERTa-based)
ROBERTA_MODEL = 'facebook/roberta-hate-speech-dynabench-r4-target'

# Hindi models (MuRIL-based, mDeBERTa for zero-shot)
MURIL_MODEL = 'google/muril-base-cased'
MDEBERTA_MODEL = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'

# Update these paths to your Google Drive
# MODEL_DIR = '/content/drive/MyDrive/Colab Notebooks/COL8381/Experiments/all_models'
MODEL_DIR  = '/kaggle/input/datasets/sansahaiiitd/all-models/all_models'

# ─────────────────────────────────────────────────────────────────────────
# ENGLISH MODELS (all use RoBERTa backbone)
# ─────────────────────────────────────────────────────────────────────────
ENGLISH_MODELS = {
    'roberta_uli_zeroshot': {
        'pth_path': f'{MODEL_DIR}/roberta_uli_zeroshot.pth',
        'base_model': ROBERTA_MODEL,
        'type': 'text-classification',
        'description': 'SOTA model | No finetuning | Indian data',
    },
    'roberta_uli_finetuned': {
        'pth_path': f'{MODEL_DIR}/roberta_uli_finetuned.pth',
        'base_model': ROBERTA_MODEL,
        'type': 'text-classification',
        'description': 'Fine-tuned on ULI | Indian data',
    },
    'roberta_uli_debiased': {
        'pth_path': f'{MODEL_DIR}/roberta_uli_debiased.pth',
        'base_model': ROBERTA_MODEL,
        'type': 'text-classification',
        'description': 'Fine-tuned + debiased on ULI | Indian data',
    },
    'roberta_west_zeroshot': {
        'pth_path': f'{MODEL_DIR}/roberta_west_zeroshot.pth',
        'base_model': ROBERTA_MODEL,
        'type': 'text-classification',
        'description': 'SOTA model | No finetuning | Western data',
    },
    'roberta_west_finetuned': {
        'pth_path': f'{MODEL_DIR}/roberta_west_finetuned.pth',
        'base_model': ROBERTA_MODEL,
        'type': 'text-classification',
        'description': 'Fine-tuned on HateXplain | Western data',
    },
}

# ─────────────────────────────────────────────────────────────────────────
# HINDI MODELS (MuRIL fine-tuned + mDeBERTa zero-shot)
# ─────────────────────────────────────────────────────────────────────────
HINDI_MODELS = {
    'muril_hindi_zeroshot': {
        'model_dir': f'{MODEL_DIR}/muril_hindi_zeroshot_model',
        'base_model': MDEBERTA_MODEL,
        'type': 'zero-shot-classification',
        'description': 'mDeBERTa NLI | Zero-shot | Hindi data',
        'candidate_labels': ['hate speech', 'not hate speech'],
    },
    'muril_hindi_finetuned': {
        'model_dir': f'{MODEL_DIR}/muril_hindi_finetuned_model',
        'base_model': MURIL_MODEL,
        'type': 'text-classification',
        'description': 'MuRIL fine-tuned | Hindi data',
    },
}

print('\n✓ Configuration loaded')

Device: GPU

✓ Configuration loaded


In [4]:
# ── LOAD ENGLISH MODELS (.pth format) ───────────────────────────────────
print('\n--- Loading English Models (RoBERTa) ---')
loaded_english = {}
loaded_english_pipes = {}

for name, config in ENGLISH_MODELS.items():
    try:
        print(f'Loading {name}...')
        model = AutoModelForSequenceClassification.from_pretrained(
            config['base_model'], num_labels=2
        )
        state = torch.load(config['pth_path'], map_location='cpu')
        model.load_state_dict(state)
        model.eval()
        if DEVICE == 0:
            model = model.cuda()
        
        tokenizer = AutoTokenizer.from_pretrained(config['base_model'])
        loaded_english[name] = model
        loaded_english_pipes[name] = pipeline(
            config['type'],
            model=model,
            tokenizer=tokenizer,
            device=DEVICE,
            batch_size=32
        )
        print(f'  ✓ {name} ready')
    except Exception as e:
        print(f'  ✗ Error loading {name}: {str(e)}')

print(f'\n✓ Loaded {len(loaded_english_pipes)} English models')


--- Loading English Models (RoBERTa) ---
Loading roberta_uli_zeroshot...


config.json:   0%|          | 0.00/816 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: facebook/roberta-hate-speech-dynabench-r4-target
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

RobertaForSequenceClassification LOAD REPORT from: facebook/roberta-hate-speech-dynabench-r4-target
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ roberta_west_zeroshot ready
Loading roberta_west_finetuned...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: facebook/roberta-hate-speech-dynabench-r4-target
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ roberta_west_finetuned ready

✓ Loaded 5 English models


In [5]:
# ── LOAD HINDI MODELS (save_pretrained format) ──────────────────────────
print('\n--- Loading Hindi Models (MuRIL + mDeBERTa) ---')
loaded_hindi = {}
loaded_hindi_pipes = {}

for name, config in HINDI_MODELS.items():
    try:
        print(f'Loading {name}...')
        model = AutoModelForSequenceClassification.from_pretrained(
            config['model_dir']
        )
        tokenizer = AutoTokenizer.from_pretrained(config['model_dir'])
        model.eval()
        if DEVICE == 0:
            model = model.cuda()
        
        loaded_hindi[name] = model
        
        # Store candidate labels for zero-shot models
        pipe = pipeline(
            config['type'],
            model=model,
            tokenizer=tokenizer,
            device=DEVICE,
            batch_size=32
        )
        loaded_hindi_pipes[name] = {
            'pipe': pipe,
            'candidate_labels': config.get('candidate_labels', None)
        }
        print(f'  ✓ {name} ready')
    except Exception as e:
        print(f'  ✗ Error loading {name}: {str(e)}')

print(f'\n✓ Loaded {len(loaded_hindi_pipes)} Hindi models')
print(f'\n=== TOTAL: {len(loaded_english_pipes) + len(loaded_hindi_pipes)} models ready ===')


--- Loading Hindi Models (MuRIL + mDeBERTa) ---
Loading muril_hindi_zeroshot...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  ✓ muril_hindi_zeroshot ready
Loading muril_hindi_finetuned...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  ✓ muril_hindi_finetuned ready

✓ Loaded 2 Hindi models

=== TOTAL: 7 models ready ===


In [6]:
# ── HELPER FUNCTIONS FOR INFERENCE & EXPLAINABILITY ────────────────────
lime_explainer = LimeTextExplainer(class_names=['Not Hate', 'Hate'])


def make_predictor_english(pipe):
    """Wrapper for text-classification pipelines (English models)."""
    def predictor(texts):
        results = pipe(texts, truncation=True)
        probs = []
        for r in results:
            if r['label'] == 'nothate':
                probs.append([r['score'], 1 - r['score']])
            else:
                probs.append([1 - r['score'], r['score']])
        return np.array(probs)
    return predictor


def make_predictor_hindi_textclass(pipe):
    """Wrapper for text-classification pipelines (Hindi fine-tuned)."""
    def predictor(texts):
        results = pipe(list(texts), truncation=True)
        probs = []
        for r in results:
            # MuRIL output: LABEL_0 = not hate, LABEL_1 = hate
            hate_score = r['score'] if r['label'] == 'LABEL_1' else 1 - r['score']
            probs.append([1 - hate_score, hate_score])
        return np.array(probs)
    return predictor


def make_predictor_hindi_zeroshot(pipe, candidate_labels):
    """Wrapper for zero-shot-classification pipelines (mDeBERTa)."""
    def predictor(texts):
        results = pipe(list(texts), candidate_labels=candidate_labels, truncation=True)
        probs = []
        for r in results:
            hate_idx = r['labels'].index('hate speech')
            hate_prob = r['scores'][hate_idx]
            probs.append([1 - hate_prob, hate_prob])
        return np.array(probs)
    return predictor


def get_verdict_english(pipe, text):
    """Get hate/not-hate verdict for English models."""
    result = pipe([text], truncation=True)[0]
    is_hate = result['label'] == 'hate'
    confidence = result['score'] if is_hate else 1 - result['score']
    not_hate_conf = 1 - confidence if is_hate else result['score']
    return is_hate, round(confidence * 100, 1), round(not_hate_conf * 100, 1)


def get_verdict_hindi_textclass(pipe, text):
    """Get hate/not-hate verdict for Hindi text-classification."""
    result = pipe([text], truncation=True)[0]
    is_hate = result['label'] == 'LABEL_1'
    confidence = result['score'] if is_hate else 1 - result['score']
    not_hate_conf = 1 - confidence if is_hate else result['score']
    return is_hate, round(confidence * 100, 1), round(not_hate_conf * 100, 1)


def get_verdict_hindi_zeroshot(pipe, text, candidate_labels):
    """Get hate/not-hate verdict for Hindi zero-shot."""
    result = pipe([text], candidate_labels=candidate_labels, truncation=True)[0]
    hate_idx = result['labels'].index('hate speech')
    is_hate = hate_idx == 0  # 'hate speech' is the top label
    hate_conf = result['scores'][hate_idx]
    confidence = hate_conf if is_hate else 1 - hate_conf
    not_hate_conf = 1 - confidence if is_hate else confidence
    return is_hate, round(confidence * 100, 1), round(not_hate_conf * 100, 1)


def get_lime_words(predictor_fn, text, n_features=8):
    """Extract LIME explanations for given text."""
    try:
        exp = lime_explainer.explain_instance(
            text, predictor_fn, num_features=n_features, num_samples=200
        )
        words = exp.as_list(label=1)  # Label 1 = Hate
        return sorted(words, key=lambda x: x[1], reverse=True)
    except:
        return []


print('✓ Helper functions defined')

✓ Helper functions defined


In [7]:
# ── BUILD HTML VISUALIZATIONS ──────────────────────────────────────────

def build_bar_chart_html(lime_words, model_name, is_hate, confidence):
    """Generate HTML card with verdict + LIME bar chart."""
    verdict_color = '#E24B4A' if is_hate else '#1D9E75'
    verdict_text = 'HATE' if is_hate else 'NOT HATE'
    
    # Get description based on model name
    desc = ''
    if model_name in ENGLISH_MODELS:
        desc = ENGLISH_MODELS[model_name]['description']
    elif model_name in HINDI_MODELS:
        desc = HINDI_MODELS[model_name]['description']
    
    max_score = max(abs(w[1]) for w in lime_words) if lime_words else 1
    bars_html = ''
    
    if lime_words:
        for word, score in lime_words:
            pct = min(abs(score) / max_score * 100, 100)
            color = '#E24B4A' if score > 0 else '#378ADD'
            label = f'+{score:.3f}' if score > 0 else f'{score:.3f}'
            bars_html += f'''
            <div style="display:flex;align-items:center;margin:4px 0;gap:8px">
              <div style="width:110px;text-align:right;font-size:13px;
                          font-family:monospace;color:#333;white-space:nowrap;
                          overflow:hidden;text-overflow:ellipsis" title="{word}">{word}</div>
              <div style="flex:1;background:#f0f0f0;border-radius:3px;height:20px">
                <div style="width:{pct}%;background:{color};height:100%;border-radius:3px;min-width:3px"></div>
              </div>
              <div style="width:55px;font-size:12px;color:{color};font-family:monospace">{label}</div>
            </div>'''
    else:
        bars_html = '<p style="color:#999;font-size:12px;margin:8px 0;">No words to explain</p>'
    
    legend = '''
    <div style="display:flex;gap:16px;margin-top:10px;font-size:11px;color:#333;font-weight:500">
      <span style="color:#888"><span style="display:inline-block;width:12px;height:12px;
            background:#E24B4A;border-radius:2px;margin-right:4px"></span>Pushes toward Hate</span>
      <span style="color:#888"><span style="display:inline-block;width:12px;height:12px;
            background:#378ADD;border-radius:2px;margin-right:4px"></span>Pushes toward Not Hate</span>
    </div>'''
    
    return f'''
    <div style="border:1.5px solid {verdict_color};border-radius:10px;
                padding:16px;margin:8px 0;background:#fff;">
      <div style="font-size:13px;font-weight:600;color:#222;font-family:monospace">{model_name}</div>
      <div style="font-size:11px;color:#777;margin-bottom:10px">{desc}</div>
      <div style="display:flex;align-items:center;gap:12px;margin-bottom:14px">
        <div style="background:{verdict_color};color:#fff;padding:6px 18px;
                    border-radius:20px;font-size:15px;font-weight:700;letter-spacing:1px">{verdict_text}</div>
        <div style="font-size:13px;color:#555;">
          <strong style="color:{verdict_color}">{confidence}%</strong> confident</div>
      </div>
      <div style="font-size:12px;color:#888;margin-bottom:8px;
                  text-transform:uppercase;letter-spacing:0.5px">Word Contributions (LIME)</div>
      {bars_html}{legend}
    </div>'''


print('✓ HTML builder defined')

✓ HTML builder defined


In [8]:
# ── MAIN PREDICTION FUNCTION ───────────────────────────────────────────

def predict(
    text,
    # English checkboxes
    # en_uli_zs, en_uli_ft, en_uli_db, en_west_zs, en_west_ft,
    en_uli_zs, en_uli_ft, en_west_zs, en_west_ft,
    # Hindi checkboxes
    hi_muril_zs, hi_muril_ft
):
    if not text or not text.strip():
        return (
            '<p style="color:#999;text-align:center">Please enter a sentence.</p>',
            '<p style="color:#999;text-align:center">Please enter a sentence.</p>'
        )
    
    # ───────────────────────────────────────────────────────────────────────
    # ENGLISH MODELS
    # ───────────────────────────────────────────────────────────────────────
    english_results = []
    
    english_checks = [
        ('roberta_uli_zeroshot', en_uli_zs),
        ('roberta_uli_finetuned', en_uli_ft),
        # ('roberta_uli_debiased', en_uli_db),
        ('roberta_west_zeroshot', en_west_zs),
        ('roberta_west_finetuned', en_west_ft),
    ]
    
    for name, is_selected in english_checks:
        if is_selected and name in loaded_english_pipes:
            try:
                pipe = loaded_english_pipes[name]
                is_hate, conf, _ = get_verdict_english(pipe, text)
                predictor = make_predictor_english(pipe)
                lime_words = get_lime_words(predictor, text)
                html = build_bar_chart_html(lime_words, name, is_hate, conf)
                english_results.append(html)
            except Exception as e:
                english_results.append(
                    f'<div style=\"color:#e74c3c;padding:16px;border-radius:10px;\">Error with {name}: {str(e)}</div>'
                )
    
    # ───────────────────────────────────────────────────────────────────────
    # HINDI MODELS
    # ───────────────────────────────────────────────────────────────────────
    hindi_results = []
    
    hindi_checks = [
        ('muril_hindi_zeroshot', hi_muril_zs),
        ('muril_hindi_finetuned', hi_muril_ft),
    ]
    
    for name, is_selected in hindi_checks:
        if is_selected and name in loaded_hindi_pipes:
            try:
                pipe_info = loaded_hindi_pipes[name]
                pipe = pipe_info['pipe']
                
                # Check if zero-shot or text-classification
                if name == 'muril_hindi_zeroshot':
                    candidate_labels = pipe_info['candidate_labels']
                    is_hate, conf, _ = get_verdict_hindi_zeroshot(pipe, text, candidate_labels)
                    predictor = make_predictor_hindi_zeroshot(pipe, candidate_labels)
                else:
                    # text-classification
                    is_hate, conf, _ = get_verdict_hindi_textclass(pipe, text)
                    predictor = make_predictor_hindi_textclass(pipe)
                
                lime_words = get_lime_words(predictor, text)
                html = build_bar_chart_html(lime_words, name, is_hate, conf)
                hindi_results.append(html)
            except Exception as e:
                hindi_results.append(
                    f'<div style=\"color:#e74c3c;padding:16px;border-radius:10px;\">Error with {name}: {str(e)}</div>'
                )
    
    # ───────────────────────────────────────────────────────────────────────
    # COMBINE RESULTS
    # ───────────────────────────────────────────────────────────────────────
    english_html = '\n'.join(english_results) if english_results else \
        '<div style="color:#999; padding:20px; text-align:center; border:1px dashed #ccc; border-radius:10px;">No English model selected</div>'
    
    hindi_html = '\n'.join(hindi_results) if hindi_results else \
        '<div style="color:#999; padding:20px; text-align:center; border:1px dashed #ccc; border-radius:10px;">No Hindi model selected</div>'
    
    return english_html, hindi_html


print('✓ Prediction function defined')

✓ Prediction function defined


In [9]:
# ── BUILD GRADIO INTERFACE ─────────────────────────────────────────────

custom_theme = gr.themes.Soft(
    primary_hue="blue",
    font=[gr.themes.GoogleFont("Poppins"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("Fira Code"), "ui-monospace", "monospace"],
)

css = """
::-webkit-scrollbar { display: none; }
* { -ms-overflow-style: none; scrollbar-width: none; }
"""

with gr.Blocks(title='Hate Speech Bias Detection Demo', theme=custom_theme, css=css) as demo:
    
    gr.Markdown("""
    # 🔍 Hate Speech Bias Detection — English + Hindi
    **Multilingual Interpretability Demo** | Comparing zero-shot, fine-tuned, and debiased models
    """)
    
    with gr.Row():
        text_input = gr.Textbox(
            label='Input Sentence (English or Hindi)',
            placeholder='Type a sentence here... (या यहाँ हिंदी में लिखें)',
            lines=3,
            scale=4
        )
    
    gr.Markdown('### Select Models')
    
    with gr.Row():
        # ───────────────────────────────────────────────────────────────────
        # ENGLISH MODELS
        # ───────────────────────────────────────────────────────────────────
        with gr.Column():
            gr.Markdown('#### English Models (RoBERTa)')
            gr.Markdown('**Indian Dataset (ULI)**')
            cb_en_uli_zs = gr.Checkbox(
                label='roberta_uli_zeroshot',
                value=True
            )
            cb_en_uli_ft = gr.Checkbox(
                label='roberta_uli_finetuned',
                value=True
            )
            # cb_en_uli_db = gr.Checkbox(
            #     label='roberta_uli_debiased',
            #     value=False
            # )
            
            gr.Markdown('**Western Dataset (HateXplain)**')
            cb_en_west_zs = gr.Checkbox(
                label='roberta_west_zeroshot',
                value=False
            )
            cb_en_west_ft = gr.Checkbox(
                label='roberta_west_finetuned',
                value=False
            )
        
        # ───────────────────────────────────────────────────────────────────
        # HINDI MODELS
        # ───────────────────────────────────────────────────────────────────
        with gr.Column():
            gr.Markdown('#### Hindi Models (MuRIL + mDeBERTa)')
            gr.Markdown('**Trained on ULI Hindi data**')
            cb_hi_muril_zs = gr.Checkbox(
                label='muril_hindi_zeroshot (mDeBERTa NLI)',
                value=True
            )
            cb_hi_muril_ft = gr.Checkbox(
                label='muril_hindi_finetuned (MuRIL)',
                value=True
            )
    
    predict_btn = gr.Button('Analyse', variant='primary', size='lg')
    
    gr.Markdown('### Results Comparison')
    
    with gr.Row():
        with gr.Column():
            gr.Markdown('#### English Model Results')
            en_output = gr.HTML()
        with gr.Column():
            gr.Markdown('#### Hindi Model Results')
            hi_output = gr.HTML()
    
    gr.Markdown("""
    ---
    **How to read each card:**
    - 🔴 **Red label** = Model predicts HATE speech
    - 🟢 **Green label** = Model predicts NOT HATE
    - **LIME bar chart** shows which words push the prediction:
      - 🔴 Red bars = words pushing toward hate
      - 🔵 Blue bars = words pushing toward not hate
    """)
    
    gr.Examples(
        examples=[
            ['Women should stay in the kitchen and not interfere in politics.'],
            ['महिलाओं को रसोई में रहना चाहिए और राजनीति में हस्तक्षेप नहीं करना चाहिए।'],
            ['Muslims are destroying the culture of this country.'],
            ['मुस्लिम इस देश की संस्कृति को नष्ट कर रहे हैं।'],
            ['Dalits are being exploited by the upper caste system even today.'],
            ['दलितों का आज भी ऊपरी जाति व्यवस्था द्वारा शोषण किया जा रहा है।'],
            ['This politician has done great work for the development of minorities.'],
            ['इस राजनेता ने अल्पसंख्यकों के विकास के लिए बेहतरीन काम किया है।'],
            ['These immigrants are stealing jobs from our people and must be deported.'],
            ['ये प्रवासी हमारे लोगों से नौकरियां चोरी कर रहे हैं और उन्हें निर्वासित किया जाना चाहिए।'],
        ],
        inputs=text_input,
        label='📌 Try these examples'
    )
    
    # Wire up the button
    predict_btn.click(
        fn=predict,
        inputs=[
            text_input,
            # English
            # cb_en_uli_zs, cb_en_uli_ft, cb_en_uli_db, cb_en_west_zs, cb_en_west_ft,
            cb_en_uli_zs, cb_en_uli_ft, cb_en_west_zs, cb_en_west_ft,
            # Hindi
            cb_hi_muril_zs, cb_hi_muril_ft
        ],
        outputs=[en_output, hi_output]
    )

print('\n✓ Gradio interface built')
print('\n' + '='*60)
print('Launching demo...')
print('='*60)

demo.launch(share=True, debug=False)
print('\n✓ Demo is live! Use the public URL above.')


✓ Gradio interface built

Launching demo...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6219bb92698d71aa95.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✓ Demo is live! Use the public URL above.
